# V2 — Evaluation (Kaggle)

Stage 2: post-processing + HumanEval on an existing `raw_responses.jsonl`.

Can run on CPU (no GPU). Copy a run from a previous session into `runs/`, or use output from generation in the same session.

In [ ]:
import os
import shutil
from pathlib import Path

INPUT_DIR = Path('/kaggle/input/datasets/kamilml/sources')
PROJECT_DIR = Path('/kaggle/working/Steganography_benchmarks_V2')
SCRIPTS_SRC = INPUT_DIR / 'scripts' if (INPUT_DIR / 'scripts').is_dir() else INPUT_DIR

EVAL_FILES = [
    'evaluate_responses.py',
    'common.py',
    'code_extract.py',
    'raw_store.py',
    'humaneval_subset_eval.py',
    'notebook_runner.py',
    'eval_handlers.py',
    'dummy_processor.py',
    'prompts.py',
    'model_runtime.py',
    'perplexity_metrics.py',
    'binoculars_scorer.py',
    'requirements.txt',
]

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
(PROJECT_DIR / 'scripts').mkdir(parents=True, exist_ok=True)

for name in EVAL_FILES:
    src = SCRIPTS_SRC / name
    if src.exists():
        shutil.copy2(src, PROJECT_DIR / 'scripts' / name)
        print(f'copied scripts/{name}')

os.chdir(PROJECT_DIR)


In [ ]:
!pip install -q human-eval tqdm

In [ ]:
# --- CONFIG ---
# If generation was in the same session, the run is already under runs/
RUN_DIR = 'runs/2026-07-11_12-00-00_gemma_humaneval_th0_0'  # <- change
DRY_RUN = False

import sys
sys.path.insert(0, 'scripts')
from notebook_runner import run_live

cmd = ['python', 'scripts/evaluate_responses.py', '--run-dir', RUN_DIR]
if DRY_RUN:
    cmd.append('--dry-run')
run_live(cmd)

In [ ]:
from pathlib import Path
import json

eval_dir = Path(RUN_DIR) / 'evaluation'
for path in sorted(eval_dir.glob('*')):
    print(path.name)

results_path = eval_dir / 'humaneval_results.json'
if results_path.exists():
    data = json.loads(results_path.read_text())
    print(f"\nPass@1: {data['passed_count']}/{data['total_count']} ({data['pass_at_1']:.1%})")